## RNN을 사용한 문장 생성

* 언어 모델을 활용해 문장 생성을 수행
    * 말뭉치를 사용해 학습한 언어 모델을 이용하여 새로운 문장을 만들어냄
    * 이후 개선된 언어 모델을 이용해 더 자연스러운 문장을 생성

* seq2seq
    * sequence to sequence
    * 한 시계열 데이터를 다른 시계열 데이터로 변환

### 언어 모델을 사용한 문장 생성

#### RNN을 사용한 문장 생성 순서
* 말뭉치로 학습한 언어 모델에서 단어를 입력으로 줌
* 언어 모델은 확률분포를 출력
    * 확률이 높은 단어를 선택하는 방법
* 특정 단어가 확률적으로 선택됨
* 앞의 작업을 되풀이
    * 선택된 단어를 다시 모델에 입력해 다음 단어의 확률분포를 얻음
* 원하는 만큼 반복  
=> 새로운 문장 생성!

* 생성한 문장은 훈련 데이터에는 존재하지 않는 말 그대로 새로 생성된 문장  
-> 훈련 데이터를 암기한 것이 아닌 훈련 데이터에서 사용된 단어의 정렬 패턴을 학습한 것
* 만약 언어 모델이 말뭉치로부터 단어의 출현 패턴을 올바르게 학습할 수 있다면  
-> 모델이 새로 생성하는 문장은 자연스럽고 의미가 통하는 문장일 것으로 기대 가능!

#### 문장 생성 구현
* Rnnlm 클래스를 상속해 RnnlmGen 클래스를 만들고 클래스에 문장 생성 메서드를 추가

In [5]:
import sys
sys.path.append('..')
import numpy as np
from common.functions import softmax
from rnnlm import Rnnlm
from better_rnnlm import BetterRnnlm


class RnnlmGen(Rnnlm):
    def generate(self, start_id, skip_ids=None, sample_size=100):
        word_ids = [start_id]

        x = start_id
        while len(word_ids) < sample_size:
            x = np.array(x).reshape(1, 1)
            score = self.predict(x)
            p = softmax(score.flatten())

            sampled = np.random.choice(len(p), size=1, p=p)
            if (skip_ids is None) or (sampled not in skip_ids):
                x = sampled
                word_ids.append(int(x.item()))

        return word_ids

    def get_state(self):
        return self.lstm_layer.h, self.lstm_layer.c

    def set_state(self, state):
        self.lstm_layer.set_state(*state)

* start_id: 최초로 주는 단어의 ID
* sample_size: 샘플링 하는 단어 수
* skip_ids: 단어 ID의 리스트, 해당 리스트에 속한 단어 ID는 샘플링되지 않도록 해줌

* 가장 먼저 model.predict(x)를 호출해 각 단어의 점수를 출력
* 점수들을 소프트맥스 함수를 이용해 정규화함
* 확률분포 p로부터 다음 단어를 샘플링
    * np.random.choice() 사용

In [6]:
import sys
sys.path.append('..')
from dataset import ptb


corpus, word_to_id, id_to_word = ptb.load_data('train')
vocab_size = len(word_to_id)
corpus_size = len(corpus)

model = RnnlmGen()
# model.load_params('Rnnlm.pkl')

# start 문자와 skip 문자 설정
start_word = 'you'
start_id = word_to_id[start_word]
skip_words = ['N', '<unk>', '$']
skip_ids = [word_to_id[w] for w in skip_words]
# 문장 생성
word_ids = model.generate(start_id, skip_ids)
txt = ' '.join([id_to_word[i] for i in word_ids])
txt = txt.replace(' <eos>', '.\n')
print(txt)

you large hide involvement warning gin frequent 12-year vegas altered fleischmann somewhere pictures unable sen. acceptable fluctuations mid-1970s shere ships board deciding radical criticism kuala n.j. lexington elaborate higher voluntary mo. bill steer concentration investment increase funds improves virtue corporate relocation credits publicity seed part establish fresenius popularity dayton alternatively generates prosecution unlikely westridge credited haas centerpiece outcry surprises psychology southwestern await yellow foster bradstreet changing improved deutsche peddling building filters accusations connected southam lot managers warner dan strengths contain miss restrict folks searle tharp caterpillar stress 1920s marcos myself heightened peasant disproportionate brokers significantly inouye screen authorities crowded aides


* 모델의 가중치 초깃값으로 무작위 값을 사용해서 의미가 통하지 않는 문장이 출력
* Rnnlm.pkl 파일로 가중치 매개변수를 읽은 후 다시 생성

In [7]:
corpus, word_to_id, id_to_word = ptb.load_data('train')
vocab_size = len(word_to_id)
corpus_size = len(corpus)

model = RnnlmGen()
model.load_params('Rnnlm.pkl')

# start 문자와 skip 문자 설정
start_word = 'you'
start_id = word_to_id[start_word]
skip_words = ['N', '<unk>', '$']
skip_ids = [word_to_id[w] for w in skip_words]
# 문장 생성
word_ids = model.generate(start_id, skip_ids)
txt = ' '.join([id_to_word[i] for i in word_ids])
txt = txt.replace(' <eos>', '.\n')
print(txt)

c:\Users\기현\Desktop\cs\deep_learning\deep_learning_from_scratch_2\common\base_model.py:42: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  params = pickle.load(f)


you need liquidity to your security chemical.
 the receiver is based on a higher cost involving money or mr. enjoyed.
 nine he says tumbled for the industry account for regular bond prices and health concerns can match negative again.
 but that news there is no competition for the competitors said one pharmaceutical trader at the warner institute of aircraft at planned research firm.
 in pacific segments of shearson filed since the ranges of those longer-term services and a write-downs of financially forced to purchase any back pressure for political swings.
 therefore did not however even


* 문법적으로 이상하거나 의미가 통하지 않는 문장이 섞여 있으나 그럴듯한 문장도 존재

#### 더 좋은 문장으로
* 더 좋은 언어 모델을 활용하면 더 좋은 문장을 기대할 수 있음!

In [11]:
corpus, word_to_id, id_to_word = ptb.load_data('train')
vocab_size = len(word_to_id)
corpus_size = len(corpus)


model = BetterRnnlmGen()
model.load_params('BetterRnnlm.pkl')

# start 문자와 skip 문자 설정
start_word = 'you'
start_id = word_to_id[start_word]
skip_words = ['N', '<unk>', '$']
skip_ids = [word_to_id[w] for w in skip_words]
# 문장 생성
word_ids = model.generate(start_id, skip_ids)
txt = ' '.join([id_to_word[i] for i in word_ids])
txt = txt.replace(' <eos>', '.\n')

print(txt)


model.reset_state()

start_words = 'the meaning of life is'
start_ids = [word_to_id[w] for w in start_words.split(' ')]

for x in start_ids[:-1]:
    x = np.array(x).reshape(1, 1)
    model.predict(x)

word_ids = model.generate(start_ids[-1], skip_ids)
word_ids = start_ids[:-1] + word_ids
txt = ' '.join([id_to_word[i] for i in word_ids])
txt = txt.replace(' <eos>', '.\n')
print('-' * 50)
print(txt)

c:\Users\기현\Desktop\cs\deep_learning\deep_learning_from_scratch_2\common\base_model.py:42: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  params = pickle.load(f)


you can ignore.
 i 'm pleased on my programs.
 the abortion effort currently have moved them from the themes of the new environmentalism law instead of a health solicitation in will slow recognition.
 in the letter states corporation assigned an office of team and community donations a so relevant to the use of consumer systems from her to two years ago.
 el espectador nev. v. los angeles miami school attorney general 's office filed the court charging enterprise business owners except to make the temptation to win few of the assumptions.
 company executives who want
--------------------------------------------------
the meaning of life is among the powerful arrangements in the maintaining competition of the brisk and global metal market.
 rather welcome western programs.
 becoming the helm of the move the u.s. seabrook usa unit hired a supreme court jury in manhattan against independent of newport beach born for the last five years if the baby boomers planned that caused the assassina

### seq2seq

#### seq2seq 원리
* Encoder-Decoder 모델
    * Encoder는 입력 데이터를 인코딩
    * Decoder는 인코딩된 데이터를 디코딩
* seq2seq는 Encoder와 Decoder가 시계열 데이터를 변환

* 시작 문장을 인코딩
* 인코딩한 정보를 Decoder에 전달
* Decoder가 도착어 문장을 생성
    * Encoder가 인코딩한 정보에는 번역에 포함된 정보가 포함됨

* Encoder는 RNN을 이용해 시계열 데이터를 h라는 은닉 상태 벡터로 변환
    * h는 LSTM 계층의 마지막 은닉 상태
    * h에 입력문장을 번역하는 데 필요한 정보가 인코딩됨
    * h는 고정 길이 벡터  
        -> 임의의 길이 문자을 고정 길이 벡터로 변환하는 작업

* Decoder는 앞 절의 신경망과 하나를 제외하고 완전히 같은 구성
    * LSTM 계층이 벡터 h를 입력받음

* LSTM 계층의 은닉 상태가 Encoder와 Decoder를 이어줌
* 순전파 때는 Encoder에서 인코딩된 정보가 LSTM 계층의 은닉 상태를 통해 Decoder로 전달
* 역전파 때 이를 통해 기울기가 Decoder로부터 Encoder로 전해짐

#### 시계열 데이터 변환용 장난감 문제
* 시계열 변환 문제의 예로 더하기
* 머신러닝을 평가하고자 만든 간단한 문제를 장난감 문제라고 함
* 단어가 아닌 문자로 분할하려 함
    * "57+5" -> ['5', '7', '+', '5']

#### 가변 길이 시계열 데이터
* 덧셈 문제에서는 샘플마다 데이터의 시간 방향 크기가 다름 -> 가변 길이 시계열 데이터
    * 미니배치 처리를 하려면 추가적인 노력이 필요함

* 가변 길이 시계열 데이터를 미니배치로 학습하기 위한 가장 단순한 방법은 패딩을 사용하는 것
* 모든 입력 데이터의 길이를 통일하고 남는 공간에는 의미없는 데이터(공백 등)를 채움
* 정답 데이터도 패딩을 수행해 모든 샘플 데이터의 길이를 통일함
* 질문, 정답을 구분하기 위해 출력 앞에 구분자로 _를 붙임
    * Decoder에 문자열을 생성하라고 알리는 신호

* 패딩을 적용해 데이터 크기를 통일시키면 가변 길이 시계열 데이터도 처리 가능
    * 그러나 원래 존재하지 않던 패딩용 문자까지 seq2seq가 처리
* 패딩을 적용해야 하지만 정확성이 중요하다면 seq2seq에 패딩 전용 처리를 추가해야 함
* Decoder에 입력된 데이터가 패딩이라면 손실의 결과에 반영하지 않도록 함
    * Softmax with Loss 계층에 마스크 기능을 추가해 해결할 수 있음
* Encoder에 입력된 데이터가 패딩이라면 LSTM 계층이 이전 시각의 입력을 그대로 출력하게 함  
    -> LSTM 계층은 마치 처음부터 패딩이 존재하지 않았던 것처럼 인코딩할 수 있음

#### 덧셈 데이터셋

In [ ]:
import sys
sys.path.append('..')
from dataset import sequence


(x_train, t_train), (x_test, t_test) = \
    sequence.load_data('addition.txt', seed=1984)
char_to_id, id_to_char = sequence.get_vocab()

print(x_train.shape, t_train.shape)
print(x_test.shape, t_test.shape)
# (45000, 7) (45000, 5)
# (5000, 7) (5000, 5)

print(x_train[0])
print(t_train[0])
# [ 3  0  2  0  0 11  5]
# [ 6  0 11  7  5]

print(''.join([id_to_char[c] for c in x_train[0]]))
print(''.join([id_to_char[c] for c in t_train[0]]))
# 71+118
# _189

(45000, 7) (45000, 5)
(5000, 7) (5000, 5)
[ 3  0  2  0  0 11  5]
[ 6  0 11  7  5]
71+118 
_189 


### seq2seq 구현
* 2개의 RNN을 연결한 신경망
* 두 RNN을 Encoder 클래스와 Decoder 클래스로 구현
* 이후 seq2seq 클래스를 구현

#### Encoder 클래스
* Encoder 클래스는 문자열을 받아 벡터 h로 변환
* Encoder 클래스는 Embedding 계층과 LSTM 계층으로 구성
    * Embedding 계층에서는 문자를 문자 벡터로 변환
* LSTM 계층은 시간 방향으로는 은닉 상태와 셀을 출력하고 위쪽으로는 은닉 상태만 출력
* Encoder 에서는 마지막 문자를 처리 후 LSTM 계층의 은닉 상태 h를 출력
* h가 Decoder로 전달

In [ ]:
class Encoder:
    def __init__(self, vocab_size, wordvec_size, hidden_size):
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = np.random.randn

        embed_W = (rn(V, D) / 100).astype('f')
        lstm_Wx = (rn(D, 4 * H) / np.sqrt(D)).astype('f')
        lstm_Wh = (rn(H, 4 * H))
        lstm_b = np.zeros(4 * H).astype('f')

        self.embed = TimeEmbedding(embed_W)
        self.lstm = TimeLSTM(lstm_Wx, lstm_Wh, lstm_b, stateful=False)

        self.params = self.embed.params + self.lstm.params
        self.grads = self.embed.grads  + self.lstm.grads

* 가중치 메서드에서는 가중치 매개변수를 초기화하고, 필요한 계층을 생성
* 가중치 매개변수와 기울기를 인스턴스 params와 grads 리스트에 보관

In [12]:
def forward(self, xs):
    xs = self.embed.forward(xs)
    hs = self.lstm.forward(xs)
    self.hs = hs
    return hs[:, -1, :]

def backward(self, dh):
    dhs = np.zeros_like(self.hs)
    dhs[:, -1, :] = dh

    dout = self.lstm.backward(dhs)
    dout = self.embed.backward(dout)
    return dout

* 순전파에서는 Time Embedding 계층과 Time LSTM 계층의 forward() 메서드를 호출
* Time LSTM 계층의 마지막 시각으이 은닉 상태만을 추출해 그 값을 Encoder의 forward() 메서드의 출력으로 반환
* 역전파에서는 LSTM 계층의 마지막 은닉 상태에 대하 기울기가 dh 인수로 전해짐
    * dh: Decoder가 전해주는 기울기
* 역전파 구현에서는 원소가 모두 0인 텐서 dhs를 생성하고 dh를 dhs의 해당 위치에 할당
* Time LSTM 계층과 Time Embeddding 계층의 backward() 메서드를 호출

#### Decoder 클래스
* Decoder 클래스는 Encoder 클래스가 출력한 h를 받아 목적으로 하는 다른 문자열을 출력
* 점수가 가장 높은 문자 하나만 결정적으로 선택

* argmax 노드: 최댓값을 가진 원소의 인덱스를 선택하는 노드
* Softmax 계층이 아닌 Affine 계층이 출력하는 점수가 가장 큰 문자 ID를 선택

In [13]:
class Decoder:
    def __init__(self, vocab_size, wordvec_size, hidden_size):
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = np.random.randn

        embed_W = (rn(V, D) / 100).astype('f')
        lstm_Wx = (rn(D, 4 * H) / np.sqrt(D)).astype('f')
        lstm_Wh = (rn(H, 4 * H))
        lstm_b = np.zeros(4 * H).astype('f')
        affine_W = (rn(H, V) / np.sqrt(H)).astype('f')
        affine_b = np.zeros(V).astype('f')

        self.embed = TimeEmbedding(embed_W)
        self.lstm = TimeLSTM(lstm_Wx, lstm_Wh, lstm_b, stateful=False)
        self.affine = TimeAffine(affine_W, affine_b)
        
        self.params, self.grads = [], []
        for layer in (self.embed, self.lstm, self.affine):
            self.params += layer.params
            self.grads += layer.grads
    
    def forward(self, xs, h):
        self.lstm.set_state(h)

        out = self.embed.forward(xs)
        out = self.lstm.forward(out)
        score = self.affine.forward(out)
        return score

    def backward(self, dscore):
        dout = self.affine.backward(dscore)
        dout = self.lstm.backward(dout)
        dout = self.embed.backward(dout)
        dh = self.lstm.dh
        return dh

* backward() 메서드는 위쪽의 Softmax with Loss 계층으로부터 기울기 dscore를 받아 Time Affine 계층, Time LSTM 계층, Time Embedding 계층 순서로 전파
    * Time LSTM 계층의 시간 방향으로의 기울기는 TimeLSTM 클래스의 인스턴스 변수 dh에 저장
    * dh를 꺼내 Decoder 클래스의 backward()의 출력으로 반환

In [ ]:
def generate(self, h, start_id, sample_size):
    sampled = []
    sample_id = start_id
    self.lstm.set_state(h)

    for _ in range(sample_size):
        x = np.array(sample_id).reshape((1, 1))
        out = self.embed.forward(x)
        out = self.lstm.forward(out)
        score = self.affine.forward(out)

        sample_id = np.argmax(score.flatten())
        sampled.append(int(sample_id))
    
    return sampled

* generate() 메서드
    * h: Encoder로부터 받는 은닉 상태인 h
    * start_id: 최초로 주어지는 문자 ID
    * sample_size: 생성하는 문자 수
* 문자를 1개씩 주고, Affine 계층이 출력하는 점수가 가장 큰 문자 ID를 선택하는 작업을 반복

#### Seq2seq 클래스
* Encoder와 Decoder 클래스를 연결
* Time Softmax with Loss 계층을 이용해 손실을 계산

In [ ]:
class Seq2seq(BaseModel):
    def __init__(self, vocab_size, wordvec_size, hidden_size):
        V, D, H = vocab_size, wordvec_size, hidden_size
        self.encoder = Encoder(V, D, H)
        self.decoder = Decoder(V, D, H)
        self.softmax = TimeSoftmaxWithLoss()

        self.params = self.encoder.params + self.decoder.params
        self.grads = self.encoder.grads + self.decoder.grads

    def forward(self, xs, ts):
        decoder_xs, decoder_ts = ts[:, :-1], ts[:, 1:]

        h = self.encoder.forward(xs)
        score = self.decoder.forward(decoder_xs, h)
        loss = self.softmax.forward(score, decoder_ts)
        return loss

    def backward(self, dout=1):
        dout = self.softmax.backward(dout)
        dh = self.decoder.backward(dout)
        dout = self.encoder.backward(dh)
        return dout

    def generate(self, xs, start_id, sample_size):
        h = self.encoder.forward(xs)
        sampled = self.decoder.generate(h, start_id, sample_size)
        return sampled

#### seq2seq 평가
* seq2seq의 학습
    1. 학습 데이터에서 미니배치를 선택
    2. 미니배치로부터 기울기를 계산
    3. 기울기를 사용하여 매개변수를 갱신

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
from dataset import sequence
from common.optimizer import Adam
from common.trainer import Trainer
from common.util import eval_seq2seq

# 데이터셋 읽기
(x_train, t_train), (x_test, t_test) = sequence.load_data('addition.txt')
char_to_id, id_to_char = sequence.get_vocab()

# 하이퍼파라미터 설정
vocab_size = len(char_to_id)
wordvec_size = 16
hideen_size = 128
batch_size = 128
max_epoch = 25
max_grad = 5.0

# 모델 / 옵티마이저 / 트레이너 생성
model = Seq2seq(vocab_size, wordvec_size, hideen_size)
optimizer = Adam()
trainer = Trainer(model, optimizer)

acc_list = []
for epoch in range(max_epoch):
    trainer.fit(x_train, t_train, max_epoch=1,
                batch_size=batch_size, max_grad=max_grad)

    correct_num = 0
    for i in range(len(x_test)):
        question, correct = x_test[[i]], t_test[[i]]
        verbose = i < 10
        correct_num += eval_seq2seq(model, question, correct,
                                    id_to_char, verbose)

    acc = float(correct_num) / len(x_test)
    acc_list.append(acc)
    print('검증 정확도 %.3f%%' % (acc * 100))

* 평가 척도로 정답률 사용
* 문자열을 생성하게 하여 그것이 답과 같은지 판정
* 맞으면 1, 틀리면 0 리턴
* 학습에 따라 정답률이 상승

### seq2seq 개선
* seq2seq를 세분화하여 학습 속도를 개선

#### 입력 데이터 반전(Reverse)
* 입력 데이터의 순서를 반전시키는 것
* 많은 경우 학습 진행이 빨라져 결과적으로 최종 정확도도 좋아짐

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
from dataset import sequence
from common.optimizer import Adam
from common.trainer import Trainer
from common.util import eval_seq2seq

# 데이터셋 읽기
(x_train, t_train), (x_test, t_test) = sequence.load_data('addition.txt')
char_to_id, id_to_char = sequence.get_vocab()

# 입력 데이터 반전
x_train, x_test = x_train[:, ::-1], x_test[:, ::-1]

# 하이퍼파라미터 설정
vocab_size = len(char_to_id)
wordvec_size = 16
hideen_size = 128
batch_size = 128
max_epoch = 25
max_grad = 5.0

# 모델 / 옵티마이저 / 트레이너 생성
model = Seq2seq(vocab_size, wordvec_size, hideen_size)
optimizer = Adam()
trainer = Trainer(model, optimizer)

acc_list = []
for epoch in range(max_epoch):
    trainer.fit(x_train, t_train, max_epoch=1,
                batch_size=batch_size, max_grad=max_grad)

    correct_num = 0
    for i in range(len(x_test)):
        question, correct = x_test[[i]], t_test[[i]]
        verbose = i < 10
        correct_num += eval_seq2seq(model, question, correct,
                                    id_to_char, verbose)

    acc = float(correct_num) / len(x_test)
    acc_list.append(acc)
    print('검증 정확도 %.3f%%' % (acc * 100))

* 입력 데이터를 반전시킨 것만으로 학습 진행이 개선
* 직관적으로 기울기 전파가 원활해지기 때문
* 다만 입력 데이터를 반전해도 단어 사이의 평균 거리는 그대로

#### 엿보기(Peeky)
* 현재의 seq2seq는 최초 시각의 LSTM 계층만 벡터 h를 이용  
    -> h를 더욱 활용할 수는 없을까?

* 두 번째 개선안: Encoder의 출력 h를 Decoder의 다른 계층에도 전달해주는 것
* LSTM과 Affine 계층에 입력되는 벡터가 2개가 됨  
    -> concat 노드를 이용

In [9]:
class PeekyDecoder:
    def __init__(self, vocab_size, wordvec_size, hidden_size):
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = np.random.randn

        embed_W = (rn(V, D) / 100).astype('f')
        lstm_Wx = (rn(H + D, 4 * H) / np.sqrt(H + D)).astype('f')
        lstm_Wh = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
        lstm_b = np.zeros(4 * H).astype('f')
        affine_W = (rn(H + H, V) / np.sqrt(H + H)).astype('f')
        affine_b = np.zeros(V).astype('f')

        self.embed = TimeEmbedding(embed_W)
        self.lstm = TimeLSTM(lstm_Wx, lstm_Wh, lstm_b, stateful=True)
        self.affine = TimeAffine(affine_W, affine_b)

        self.params, self.grads = [], []
        for layer in (self.embed, self.lstm, self.affine):
            self.params += layer.params
            self.grads += layer.grads
        self.cache = None

    def forward(self, xs, h):
        N, T = xs.shape
        N, H = h.shape

        self.lstm.set_state(h)

        out = self.embed.forward(xs)
        hs = np.repeat(h, T, axis=0).reshape(N, T, H)
        out = np.concatenate((hs, out), axis=2)

        out = self.lstm.forward(out)
        out = np.concatenate((hs, out), axis=2)

        score = self.affine.forward(out)
        self.cache = H
        return score

    def backward(self, dscore):
        H = self.cache

        dout = self.affine.backward(dscore)
        dout, dhs0 = dout[:, :, H:], dout[:, :, :H]
        dout = self.lstm.backward(dout)
        dembed, dhs1 = dout[:, :, H:], dout[:, :, :H]
        self.embed.backward(dembed)

        dhs = dhs0 + dhs1
        dh = self.lstm.dh + np.sum(dhs, axis=1)
        return dh

    def generate(self, h, start_id, sample_size):
        sampled = []
        char_id = start_id
        self.lstm.set_state(h)

        H = h.shape[1]
        peeky_h = h.reshape(1, 1, H)
        for _ in range(sample_size):
            x = np.array([char_id]).reshape((1, 1))
            out = self.embed.forward(x)

            out = np.concatenate((peeky_h, out), axis=2)
            out = self.lstm.forward(out)
            out = np.concatenate((peeky_h, out), axis=2)
            score = self.affine.forward(out)

            char_id = np.argmax(score.flatten())
            sampled.append(char_id)

        return sampled

* Decoder의 초기화는 이전의 Decoder와 동일
* 다른 점은 LSTM 계층의 가중치, Affine 계층의 가중치의 형상
* Encoder가 인코딩한 벡터도 입력되기 때문에 가증치 매개변수의 형상이 더욱 커짐

* forward의 구현은 h를 np.repeat()로 시계열만큼 복제해 hs에 저장
* np.concatenate()를 이용해 hs와 Embedding 계층의 출력을 연결하고 LSTM 계층에 입력
* Affine 계층도 hs와 LSTM의 출력을 연결한 것을 입력

In [10]:
class PeekySeq2seq(Seq2seq):
    def __init__(self, vocab_size, wordvec_size, hidden_size):
        V, D, H = vocab_size, wordvec_size, hidden_size
        self.encoder = Encoder(V, D, H)
        self.decoder = PeekyDecoder(V, D, H)
        self.softmax = TimeSoftmaxWithLoss()

        self.params = self.encoder.params + self.decoder.params
        self.grads = self.encoder.grads + self.decoder.grads

* 입력 반전과 Peeky를 모두 적용하면 결과가 매우 좋아짐
* seq2seq의 정확도는 하이퍼파라미터에 영향을 크게 받음
* 실제 문제에서는 효과가 달라질 것

### seq2seq를 이용하는 애플리케이션
* 시계열 데이터를 반환하는 프레임워크는 다양한 문제에 적용 가능
    * 기계 번역: '한 언어의 문장'을 '다른 언어의 문장'으로 변환
    * 자동 요약: '긴 문장'을 '짧게 요약된 문장'으로 변환
    * 질의응답: '질문'을 '응답'으로 변환
    * 메일 자동 응답: '받은 메일의 문장'을 '답변 글'로 변환

* seq2seq는 2개가 짝을 이루는 시계열 데이터를 다루는 문제에 이용할 수 있음
* seq2seq가 적용될 수 없을 것 같은 문제라도 입력, 출력 데이터를 전처리하면 seq2seq를 적용할 수 있는 경우도 있음

#### 챗봇
* 챗봇은 사람과 컴퓨터가 텍스트로 대화를 나누는 프로그램
* 챗봇에도 seq2seq를 사용할 수 있음
* 대화 기반으로 정답이나 힌트를 얻는 방식은 실용성이 높고 다양하게 응용하여 효과를 볼 수 있음

#### 알고리즘 학습
* 고차원적인 문제도 처리할 수 있음 ex) 파이썬 코드
* 소스 코드 또한 문자로 쓰여진 시계열 데이터
* 일반적으로 잘 풀리지 않을 수 있으나 구조를 개선하면 문제를 풀 수 있음

#### 이미지 캡셔닝
* seq2seq는 이미지나 음성 등 다양한 데이터를 처리할 수 있음
* 이미지 캡셔닝: 이미지를 문장으로 변환
* Encoder가 LSTM에서 CNN으로 바뀜
    * 이미지의 인코딩을 CNN이 수행
    * CNN의 출력은 특징 맵이므로 이를 Decoder의 LSTM이 처리할 수 있도록 평탄화 후 완전연결인 Affine 계층에서 변환
    * 변환된 데이터를 Decoder에 전달
